In [28]:
import os
import sys
import requests
import datetime as dt
from dotenv import load_dotenv
from pathlib import Path
import requests
import pandas as pd
import talib as ta
import plotly.graph_objs as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Button, Output
from IPython.display import display

from Modules.rci import Rci

# CorporateFinanceData 動作確認
`CorporateFinanceData` を使って財務データ取得メソッドを個別セルで実行します。

In [30]:
# APIの接続先（notebookコンテナでは通常 http://api:8000 が設定される）
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
# データを取得する関数
sys.path.append("/workspace/notebook")

### エンドポイントへRequestし財務データを更新する

In [34]:
# Vestas Wind Systems (Yahoo Finance: VWS.CO)
code = "VWS"
market = "CO"

post_payload = {
    "code": code,
    "market": market,
}

# /api/v1/corp_finance_data/
post_url = f"{API_BASE_URL}/api/v1/corp_finance_data/"
post_response = requests.post(post_url, json=post_payload, timeout=60)

print("POST URL:", post_url)
print("POST status:", post_response.status_code)
print("POST response:", post_response.json())

POST URL: http://api:8000/api/v1/corp_finance_data/
POST status: 200
POST response: {'result': True}


### 直近４年分の財務諸表を取得

In [40]:
get_params = {
    "code": code,
    "market": market,
}
# /api/v1/corp_finance_data/financials/
get_url = f"{API_BASE_URL}/api/v1/corp_finance_data/financials/"
get_response = requests.get(get_url, json=get_params, timeout=60)

get_url = f"{API_BASE_URL}/api/v1/corp_finance_data/financials/"
get_response = requests.get(get_url, params=get_params, timeout=60)

financials_df = pd.DataFrame(get_response.json().get("results", []))
financials_df

,date,market,code,tax_effect_of_unusual_items,tax_rate_for_calcs,normalized_ebitda,total_unusual_items,total_unusual_items_excluding_goodwill,net_income_from_continuing_operation_net_minority_interest,reconciled_depreciation,reconciled_cost_of_revenue,ebitda,ebit,net_interest_income,interest_expense,interest_income,normalized_income,net_income_from_continuing_and_discontinued_operation,total_expenses,total_operating_income_as_reported,diluted_average_shares,basic_average_shares,diluted_eps,basic_eps,diluted_ni_availto_com_stockholders,net_income_common_stockholders,otherunder_preferred_stock_dividend,net_income,minority_interests,net_income_including_noncontrolling_interests,net_income_continuous_operations,tax_provision,pretax_income,special_income_charges,other_special_charges,write_off,impairment_of_capital_assets,net_non_operating_interest_income_expense,total_other_finance_cost,interest_expense_non_operating,interest_income_non_operating,operating_income,operating_expense,other_operating_expenses,research_and_development,selling_general_and_administration,selling_and_marketing_expense,general_and_administrative_expense,gross_profit,cost_of_revenue,total_revenue,operating_revenue,created_at,updated_at
0,2021-12-31,CO,VWS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.007971e+09,1.005048e+09,0.13,0.13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z
1,2022-12-31,CO,VWS,-32266000.0,0.073,-1.030000e+08,-442000000.0,-442000000.0,-1.572000e+09,1.088000e+09,1.436800e+10,-5.450000e+08,-1.633000e+09,-42000000.0,63000000.0,37000000.0,-1.162266e+09,-1.572000e+09,1.563800e+10,-1.596000e+09,1.006178e+09,1.006178e+09,-1.56,-1.56,-1.572000e+09,-1.572000e+09,0.0,-1.572000e+09,0.0,-1.572000e+09,-1.572000e+09,-124000000.0,-1.696000e+09,-444000000.0,115000000.0,260000000.0,69000000.0,-42000000.0,16000000.0,63000000.0,37000000.0,-1.152000e+09,1.270000e+09,87000000.0,457000000.0,8.130000e+08,462000000.0,351000000.0,1.180000e+08,1.436800e+10,1.448600e+10,1.448600e+10,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z
2,2023-12-31,CO,VWS,49742000.0,0.238,9.170000e+08,209000000.0,209000000.0,7.700000e+07,7.890000e+08,1.409900e+10,1.126000e+09,3.370000e+08,-66000000.0,235000000.0,205000000.0,-8.225800e+07,7.700000e+07,1.534600e+10,2.920000e+08,1.009800e+09,1.006360e+09,0.08,0.08,7.700000e+07,7.700000e+07,0.0,7.700000e+07,-1000000.0,7.800000e+07,7.800000e+07,24000000.0,1.020000e+08,208000000.0,-151000000.0,-45000000.0,-12000000.0,-66000000.0,36000000.0,235000000.0,205000000.0,3.600000e+07,1.247000e+09,NaN,371000000.0,8.760000e+08,452000000.0,424000000.0,1.283000e+09,1.409900e+10,1.538200e+10,1.538200e+10,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z
3,2024-12-31,CO,VWS,17100000.0,0.300,1.704000e+09,57000000.0,57000000.0,4.990000e+08,8.620000e+08,1.523800e+10,1.761000e+09,8.990000e+08,-70000000.0,194000000.0,157000000.0,4.591000e+08,4.990000e+08,1.655600e+10,7.940000e+08,1.010126e+09,1.005975e+09,0.49,0.50,4.990000e+08,4.990000e+08,0.0,4.990000e+08,5000000.0,4.940000e+08,4.940000e+08,211000000.0,7.050000e+08,53000000.0,-56000000.0,1000000.0,2000000.0,-70000000.0,33000000.0,194000000.0,157000000.0,7.390000e+08,1.318000e+09,NaN,380000000.0,9.380000e+08,535000000.0,403000000.0,2.057000e+09,1.523800e+10,1.729500e+10,1.729500e+10,2026-04-15T08:05:01.086938Z,2026-04-15T08:05:01.086938Z
4,2025-12-31,CO,VWS,-16750000.0,0.250,2.310000e+09,-67000000.0,-67000000.0,7.780000e+08,1.059000e+09,1.632500e+10,2.243000e+09,1.184000e+09,17000000.0,145000000.0,195000000.0,8.282500e+08,7.780000e+08,1.775500e+10,1.015000e+09,NaN,NaN,NaN,NaN,7.780000e+08,7.780000e+08,0.0,7.780000e+08,-2000000.0,7.800000e+08,7.800000e+08,259000000.0,1.039000e+09,-52000000.0,37000000.0,-6000000.0,21000000.0,17000000.0,33000000.0,145000000.0,195000000.0,1.067000e+09,1.430000e+09,NaN,424000000.0,1.006000e+09,545000000.0,461000000.0,2.4970

### 直近４年分のバランスシートを取得

### 直近４年分のキャッシュフローを取得

### 直近４年分の収益を取得

### 直近４年分の四半期収益を取得